# Benchmark CAM latency

Compare **LayerCAM** and **SmoothGradCAM++** on torchvision **ResNet18** and **MobileNet V3 Large**. This notebook measures the CAM operation alone and the model-forward-plus-CAM operation separately, reports every sample, and plots median and interquartile range (IQR).

The default experiment runs on **CPU**, uses **random weights**, and takes **five measurements after two CAM warmups** for each combination. No model download or CUDA installation is required. Random weights are suitable for checking computational cost; their activation maps and class scores do **not** provide useful explanations. Set `USE_PRETRAINED = True` to measure pretrained models, and record that choice when sharing results.

This notebook follows the timing boundaries in the pinned [`scripts/eval_latency.py`](https://github.com/frgfm/torch-cam/blob/e311623a60474aeab359afee5c432cc97a27269e/scripts/eval_latency.py), with warmup of the actual extraction path and a small default sample count. The five-sample run is a quick experiment; use at least 100 measurements for a more stable comparison.

## Install the pinned implementation

Use Python 3.11 or newer in a fresh notebook kernel. This is the **development snapshot after TorchCAM 0.5.0**, pinned to commit `e311623a60474aeab359afee5c432cc97a27269e`; it includes later numerical and benchmark corrections. It is not the released 0.5.0 package. `%pip` installs into the current kernel's environment. If TorchCAM, PyTorch, or torchvision was already imported, restart the kernel after installation and continue below.

In [ ]:
%pip install "torchcam[scripts] @ git+https://github.com/frgfm/torch-cam.git@e311623a60474aeab359afee5c432cc97a27269e"

In [ ]:
import importlib.metadata
import json
import os
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from IPython.display import HTML, display
from torchvision.models import get_model, get_model_weights

import torchcam
from torchcam.methods import LayerCAM, SmoothGradCAMpp

TORCHCAM_REF = "e311623a60474aeab359afee5c432cc97a27269e"

## Choose the workload

Both models receive the same `(1, 3, 224, 224)` float32 random input and class index. We choose target layers explicitly: `layer4` for ResNet18 and `features` for MobileNet V3 Large. Each produces a 7×7 map at this input size; they have different channel counts and computational costs.

`INPUT_BATCH_SIZE` is the classifier's image batch size. This experiment keeps it at one. Neither method uses ScoreCAM's masked-image batching, so that batch setting is not applicable. SmoothGradCAM++ uses four noisy samples with standard deviation 0.3 per extraction; those extra forwards are part of its measured cost.

Change `DEVICE` to `"cuda:0"` for an available CUDA GPU or `"mps"` for a supported Apple GPU. Accelerator timing uses synchronization before and after each interval. Keep the same settings and hardware when comparing runs. CPU threads are set explicitly because they can strongly affect latency.

In [ ]:
DEVICE = "cpu"
CPU_THREADS = 1
SEED = 0
INPUT_BATCH_SIZE = 1
IMAGE_SIZE = 224
CLASS_IDX = 232
USE_PRETRAINED = False
WARMUP_ITERATIONS = 2
MEASURED_ITERATIONS = 5  # Set to 100 for a longer run.
SMOOTHGRAD_SAMPLES = 4
SMOOTHGRAD_STD = 0.3
SCOPES = ("extractor", "end-to-end")
MODEL_LAYERS = {
    "resnet18": "layer4",
    "mobilenet_v3_large": "features",
}
METHODS = {"LayerCAM": LayerCAM, "SmoothGradCAMpp": SmoothGradCAMpp}

device = torch.device(DEVICE)
if device.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; use DEVICE = 'cpu' or enable a CUDA runtime.")
if device.type == "mps" and not torch.backends.mps.is_available():
    raise RuntimeError("MPS is unavailable; use DEVICE = 'cpu'.")
if INPUT_BATCH_SIZE != 1:
    raise ValueError("This benchmark's fixed-class workload expects INPUT_BATCH_SIZE = 1.")
if min(CPU_THREADS, IMAGE_SIZE, WARMUP_ITERATIONS, MEASURED_ITERATIONS) <= 0:
    raise ValueError("Thread count, image size, warmups, and measurements must be positive.")
torch.set_num_threads(CPU_THREADS)
torch.manual_seed(SEED)

In [ ]:
processor = platform.processor()
if not processor and Path("/proc/cpuinfo").is_file():
    processor = next(
        (line.split(":", 1)[1].strip() for line in Path("/proc/cpuinfo").read_text().splitlines()
         if line.startswith("model name")),
        "",
    )
processor = processor or "CPU model unavailable on this platform"

distribution = importlib.metadata.distribution("torchcam")
source_metadata = json.loads(distribution.read_text("direct_url.json") or "{}")
installed_ref = source_metadata.get("vcs_info", {}).get("commit_id")
if installed_ref is not None and installed_ref != TORCHCAM_REF:
    raise RuntimeError("Installed TorchCAM commit differs from this notebook's pinned setup.")

environment = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "torchcam": torchcam.__version__,
    "numpy": np.__version__,
    "matplotlib": importlib.metadata.version("matplotlib"),
    "torchcam_requested_ref": TORCHCAM_REF,
    "torchcam_install_source": source_metadata,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": processor,
    "logical_cpu_count": os.cpu_count(),
    "torch_cpu_threads": torch.get_num_threads(),
    "device": str(device),
    "device_name": torch.cuda.get_device_name(device) if device.type == "cuda" else (processor if device.type == "cpu" else "Apple MPS"),
    "cuda_runtime": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "cudnn_benchmark": torch.backends.cudnn.benchmark,
}
settings = {
    "seed": SEED,
    "input_shape": [INPUT_BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE],
    "input_dtype": "float32",
    "input_distribution": "uniform[0,1), no image preprocessing",
    "class_idx": CLASS_IDX,
    "weights": "torchvision DEFAULT" if USE_PRETRAINED else "none (random initialization)",
    "warmups_per_combination": WARMUP_ITERATIONS,
    "measurements_per_combination": MEASURED_ITERATIONS,
    "scopes": list(SCOPES),
    "target_layers": MODEL_LAYERS,
    "smoothgrad_samples": SMOOTHGRAD_SAMPLES,
    "smoothgrad_std": SMOOTHGRAD_STD,
    "cam_masked_input_batch_size": "not applicable to these methods",
}
print(json.dumps({"environment": environment, "settings": settings}, indent=2))

## Define the timing boundaries

- **Extractor:** run one classifier forward before the timer starts, then measure `extractor(class_idx, scores)`. Additional forwards inside SmoothGradCAM++ remain inside the interval.
- **End-to-end:** measure the initial classifier forward and CAM extraction together.

Both scopes exclude model construction, weight loading, input creation and transfer, gradient reset, map validation, plotting, and any image preprocessing or file I/O. “End-to-end” here refers to the model and CAM operations, rather than an entire image-serving pipeline. CUDA/MPS completion synchronization is inside the interval; synchronization before starting removes outstanding work.

Warmups use the same scope and CAM method as the measured samples. Every iteration gets a fresh forward graph and resets model/input gradients. Context managers remove hooks between combinations. CAM shapes and finite values are checked after timing, so a fast failed extraction cannot become a benchmark result. We do not interpret or display the random-weight maps.

In [ ]:
def synchronize_device():
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps":
        torch.mps.synchronize()


def time_cam(model, input_tensor, extractor, scope):
    model.zero_grad(set_to_none=True)
    input_tensor.grad = None
    if scope == "extractor":
        scores = model(input_tensor)

    synchronize_device()
    started_at = time.perf_counter()
    if scope == "end-to-end":
        scores = model(input_tensor)
    cams = extractor(CLASS_IDX, scores)
    synchronize_device()
    elapsed_ms = 1000 * (time.perf_counter() - started_at)
    return elapsed_ms, cams


def validate_cams(cams, expected_shapes=None):
    if not cams or any(
        not isinstance(cam, torch.Tensor)
        or cam.ndim != 3
        or cam.shape[0] != INPUT_BATCH_SIZE
        or cam.numel() == 0
        or not torch.isfinite(cam).all().item()
        for cam in cams
    ):
        raise RuntimeError("Extraction returned an invalid CAM.")
    shapes = [list(cam.shape) for cam in cams]
    if expected_shapes is not None and shapes != expected_shapes:
        raise RuntimeError(f"CAM shape changed: {expected_shapes} -> {shapes}")
    return shapes


def benchmark(architecture, target_layer, method_name, extractor_cls, scope):
    # Restart the random sequence for comparable model initialization and inputs.
    torch.manual_seed(SEED)
    weights = get_model_weights(architecture).DEFAULT if USE_PRETRAINED else None
    model = get_model(architecture, weights=weights).eval().to(device=device, dtype=torch.float32)
    model.requires_grad_(False)
    input_tensor = torch.rand(
        (INPUT_BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE),
        device=device,
        dtype=torch.float32,
        requires_grad=True,
    )
    extractor_kwargs = {"target_layer": target_layer}
    if extractor_cls is SmoothGradCAMpp:
        extractor_kwargs.update(num_samples=SMOOTHGRAD_SAMPLES, std=SMOOTHGRAD_STD)

    measurements = []
    expected_shapes = None
    with extractor_cls(model, **extractor_kwargs) as extractor:
        for iteration in range(WARMUP_ITERATIONS + MEASURED_ITERATIONS):
            elapsed_ms, cams = time_cam(model, input_tensor, extractor, scope)
            expected_shapes = validate_cams(cams, expected_shapes)
            if iteration >= WARMUP_ITERATIONS:
                measurements.append(elapsed_ms)

    q1, median, q3 = np.percentile(measurements, [25, 50, 75])
    return {
        "architecture": architecture,
        "method": method_name,
        "target_layer": target_layer,
        "scope": scope,
        "weights": str(weights) if weights is not None else "none",
        "device": str(device),
        "cam_shapes": expected_shapes,
        "samples_ms": measurements,
        "q1_ms": float(q1),
        "median_ms": float(median),
        "q3_ms": float(q3),
        "iqr_ms": float(q3 - q1),
    }

In [ ]:
results = []
for architecture, target_layer in MODEL_LAYERS.items():
    for method_name, extractor_cls in METHODS.items():
        for scope in SCOPES:
            result = benchmark(architecture, target_layer, method_name, extractor_cls, scope)
            results.append(result)
            print(
                f"{architecture:<19} {method_name:<16} {scope:<11} "
                f"median={result['median_ms']:.2f} ms, IQR={result['iqr_ms']:.2f} ms"
            )

assert len(results) == len(MODEL_LAYERS) * len(METHODS) * len(SCOPES)
assert all(len(result["samples_ms"]) == MEASURED_ITERATIONS for result in results)

## Inspect the measured distributions

The table includes medians and IQRs in milliseconds. Each plot shows all samples, with a black dot for the median and a bar from the first to the third quartile. Separate plots for the two timing scopes keep their definitions visible. The latency axes use a logarithmic scale so the distributions remain visible when costs differ by orders of magnitude.

Do not infer explanation quality from latency, or treat architecture names as categorical CPU/GPU recommendations. Results depend on device, software, target layer, thread settings, sample count, and weights. These runs are sequential and an accelerator or CPU under other load can affect them. Five samples are too few to establish tail latency. Even after warmup, the difference between two medians is not an isolated measurement of the initial forward: each scope is measured in a separate run.

In [ ]:
rows = "".join(
    "<tr>"
    f"<td>{result['architecture']}</td><td>{result['method']}</td>"
    f"<td>{result['target_layer']}</td><td>{result['scope']}</td>"
    f"<td>{len(result['samples_ms'])}</td><td>{result['median_ms']:.2f}</td>"
    f"<td>{result['iqr_ms']:.2f}</td>"
    "</tr>"
    for result in results
)
display(HTML(
    "<table><thead><tr><th>Model</th><th>Method</th><th>Target layer</th><th>Scope</th>"
    "<th>Samples</th><th>Median (ms)</th><th>IQR (ms)</th></tr></thead>"
    f"<tbody>{rows}</tbody></table>"
))

In [ ]:
fig, axes = plt.subplots(1, len(SCOPES), figsize=(13, 5), squeeze=False)
for ax, scope in zip(axes[0], SCOPES):
    selected = [result for result in results if result["scope"] == scope]
    for index, result in enumerate(selected):
        samples = result["samples_ms"]
        jitter = np.linspace(-0.12, 0.12, len(samples))
        ax.scatter(index + jitter, samples, alpha=0.6, color="tab:blue", label="Samples" if index == 0 else None)
        median = result["median_ms"]
        ax.errorbar(
            index, median,
            yerr=[[median - result["q1_ms"]], [result["q3_ms"] - median]],
            fmt="o", color="black", capsize=6, label="Median and IQR" if index == 0 else None,
        )
    ax.set_xticks(range(len(selected)), [f"{r['architecture']}\n{r['method']}" for r in selected], rotation=25, ha="right")
    ax.set_title(f"Timing scope: {scope}")
    ax.set_ylabel("Latency (ms, logarithmic scale)")
    ax.set_yscale("log")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
fig.suptitle(f"TorchCAM latency · {device} · weights: {settings['weights']}")
fig.tight_layout()
plt.show()

## Keep the experiment with its results

The next cell saves the environment, exact requested code reference and install source, workload settings, target layers, actual weight-enum names, and all measured samples. Share this JSON alongside any plots. The package is pinned; other installed dependency versions are recorded rather than locked, so reproduce those versions for a strict comparison.

For a more representative run, use `MEASURED_ITERATIONS = 100`, increase warmups if the measurements are still stabilizing, and rerun from the configuration cell. To benchmark another device, change `DEVICE` and rerun the same cells. Pretrained weights are optional and require a download on first use. Their input here remains synthetic; use real preprocessed images for an experiment that also assesses attribution quality.

In [ ]:
output_path = Path("latency_results.json")
output_path.write_text(
    json.dumps({"environment": environment, "settings": settings, "results": results}, indent=2) + "\n",
    encoding="utf-8",
)
print(f"Saved {len(results)} measured combinations to {output_path.resolve()}")